In [1]:
import pandas as pd
import torch
from torch_geometric.loader import DataLoader

from scripts.data_formatting import SmilesDataset
from scripts.downstream import get_prediction_smiles, split_batch_by_molecule
from scripts.nn_models import GINE

In [ ]:
# raw dataset
df = pd.read_csv('../datasets/test.csv')
smiles_list = df['SMILES Labelled'].tolist()

In [3]:
# dataset objects
test_dataset  = SmilesDataset(smiles_list)

# dataloaders
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [4]:
# model instance
model = GINE(input_dim=9, hidden_dim=128, output_dim=5, edge_dim=8, num_layers=3)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()

In [5]:
MODEL_PATH = '../models/gine/gine_156.pt'
OUTPUT_CSV = '../results/test.csv'

# load best model
model.load_state_dict(torch.load(MODEL_PATH))

smiles_original = []
smiles_predicted = []
smiles_matched = []

# test evaluation
model.eval()
test_loss, correct, total = 0, 0, 0
with torch.no_grad():
    for batch in test_loader:
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y.argmax(dim=1))
        test_loss += loss.item()

        preds, true = split_batch_by_molecule(out, batch)
        for p, t in zip(preds, true):
            if torch.equal(p, t):
                smiles_matched.append(True)
                correct += 1
            else:
                smiles_matched.append(False)
            total += 1

        smiles_original.extend(batch.smiles)
        smiles_predicted.extend(get_prediction_smiles(preds, batch.smiles))

avg_test_loss = test_loss / len(test_loader)
test_acc = correct / total

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_acc*100:.2f}%")

# saves test predictions to .csv file
test_df = pd.DataFrame({
    'SMILES Original': smiles_original,
    'SMILES Predicted': smiles_predicted,
    'SMILES Matched': smiles_matched
})
test_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved predictions to {OUTPUT_CSV}.")

tensor(True) tensor(0.8990) tensor(0.0659)
tensor(True) tensor(0.7395) tensor(0.0360)
tensor(True) tensor(0.7582) tensor(0.0761)
tensor(True) tensor(0.7678) tensor(0.1883)
tensor(True) tensor(0.8319) tensor(0.1680)
tensor(True) tensor(0.1019) tensor(0.0069)
tensor(True) tensor(0.8738) tensor(0.0436)
tensor(False) tensor(0.0744) tensor(0.3851)
tensor(True) tensor(0.4764) tensor(0.0128)
tensor(False) tensor(0.3280) tensor(0.6719)
tensor(True) tensor(0.6670) tensor(0.0407)
tensor(True) tensor(0.9962) tensor(1.3289e-05)
tensor(True) tensor(0.9120) tensor(0.0838)
tensor(True) tensor(0.9995) tensor(0.0003)
tensor(False) tensor(0.0452) tensor(0.9548)
tensor(True) tensor(0.8599) tensor(0.1295)
tensor(True) tensor(0.9354) tensor(0.0018)
tensor(True) tensor(0.7900) tensor(0.2087)
tensor(True) tensor(0.8692) tensor(0.1307)
tensor(True) tensor(0.9989) tensor(0.0009)
tensor(True) tensor(0.1107) tensor(0.0109)
tensor(True) tensor(0.7122) tensor(0.0011)
tensor(True) tensor(0.9654) tensor(0.0030)
tens

In [6]:
# OVERALL CORRECT: 64.71%
# SOURCE-SINK CORRECT: 68.51%
# 10 correct: 74.74%
# 11 correct: 87.89%
# 20 correct: 83.04%
# 21 correct: 80.97%